In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import pandas as pd
import os
import sys
import subprocess
import urllib.request
import urllib.error
import json
import time
import random
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm import tqdm

# ==========================================
# 1. 基础配置 (Configuration)
# ==========================================

class Config:
    def __init__(self):
        self.num_students = 0
        self.num_questions = 0
        self.num_concepts = 0

        # 训练超参数
        self.embed_dim = 64
        self.seq_len = 100
        self.batch_size = 64
        self.epochs = 5
        self.lr = 0.001
        self.dropout = 0.3
        self.tcan_layers = 2
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        # LLM 实验配置
        self.api_key = ""
        self.llm_model = "gemini-2.5-flash-preview-09-2025"
        self.llm_sample_size = 20
        self.showcase_count = 2

        # 数据集配置: Math2015
        self.data_dir = "./data/math2015"
        self.dataset_name = "math2015"

# ==========================================
# 2. 数据处理模块 (针对 Math2015/FrcSub 优化)
# ==========================================

class Math2015Processor:
    def __init__(self, config):
        self.config = config
        if not os.path.exists(config.data_dir):
            os.makedirs(config.data_dir)

    def _find_data_file(self, path):
        """递归寻找目录下作答数据文件和对应的 Q 矩阵文件"""
        data_file, q_file = None, None
        for root, _, files in os.walk(path):
            for file in files:
                low_name = file.lower()
                # 排除说明文件
                if any(k in low_name for k in ["termsofuse", "license", "readme", "instruction", "meta"]):
                    continue

                full_path = os.path.join(root, file)
                # 识别作答数据 (通常叫 data.txt, mat.txt, log.csv)
                if any(k in low_name for k in ["data", "mat", "log"]) and file.endswith((".txt", ".csv")):
                    data_file = full_path
                # 识别 Q 矩阵 (通常叫 q.txt, q_matrix.csv)
                if low_name.startswith("q") and file.endswith((".txt", ".csv")):
                    q_file = full_path

        return data_file, q_file

    def _download_data(self):
        """下载数据集"""
        data_path, q_path = self._find_data_file(self.config.data_dir)

        if not data_path:
            print(f"正在下载 {self.config.dataset_name}...")
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "EduData"])
                from EduData import get_data
                get_data(self.config.dataset_name, self.config.data_dir)
            except Exception as e:
                print(f"下载失败: {e}")

            data_path, q_path = self._find_data_file(self.config.data_dir)

        if not data_path:
            raise FileNotFoundError("未能找到有效的数据文件。")

        return data_path, q_path

    def process(self):
        data_path, q_path = self._download_data()
        print(f"找到数据文件: {data_path}")

        # 1. 读取作答数据
        # 尝试不带表头读取，并自动处理分隔符
        df_raw = pd.read_csv(data_path, sep=None, engine='python', header=None)

        # 判断是否为稠密矩阵格式 (列名为数字，且没有明显的文本列)
        is_dense = True
        # 如果第一列包含 'user' 等字样，说明有表头
        try:
            test_read = pd.read_csv(data_path, sep=None, engine='python', nrows=1)
            if any(isinstance(c, str) and 'user' in c.lower() for c in test_read.columns):
                is_dense = False
                df_raw = pd.read_csv(data_path, sep=None, engine='python')
        except:
            pass

        if is_dense:
            print("检测到稠密矩阵格式，正在转换为三元组...")
            # 转换为 [user_id, item_id, score]
            df = df_raw.stack().reset_index()
            df.columns = ['user_id', 'item_id', 'score']
        else:
            # 兼容性列名映射
            df = df_raw.copy()
            mapping = {
                'user_id': ['user_id', 'UID', 'u', 'user'],
                'item_id': ['item_id', 'IID', 'i', 'item', 'problem_id'],
                'score': ['score', 'label', 'r', 'correct'],
                'knowledge_code': ['knowledge_code', 'c', 'skill', 'q']
            }
            for std, alts in mapping.items():
                for alt in alts:
                    if alt in df.columns:
                        df = df.rename(columns={alt: std})
                        break

        # 2. 处理 Q 矩阵 (知识点映射)
        if q_path:
            print(f"找到 Q 矩阵文件: {q_path}")
            q_df = pd.read_csv(q_path, sep=None, engine='python', header=None)
            # 构建知识点编码列
            # 格式：对于每一道题(行)，列出值为 1 的列索引(知识点)
            item_to_skills = {}
            for i, row in q_df.iterrows():
                skills = np.where(row.values == 1)[0]
                item_to_skills[i] = ",".join(map(str, skills))

            if 'item_id' in df.columns:
                df['knowledge_code'] = df['item_id'].map(item_to_skills)
        elif 'knowledge_code' not in df.columns:
            # 如果完全没有知识点信息，生成伪知识点（不推荐，但保证可运行）
            print("警告: 未找到知识点映射，将生成默认映射。")
            df['knowledge_code'] = df['item_id'].astype(str)

        # 基础清洗
        df = df.dropna(subset=['knowledge_code', 'score'])
        df['score'] = (df['score'] >= 0.5).astype(int)

        # ID 映射
        user_map = {val: i+1 for i, val in enumerate(df['user_id'].unique())}
        item_map = {val: i+1 for i, val in enumerate(df['item_id'].unique())}

        unique_concepts = set()
        for codes in df['knowledge_code'].astype(str).unique():
            cleaned = codes.replace('[', '').replace(']', '').replace("'", "").replace('"', '')
            for code in cleaned.split(','):
                code = code.strip()
                if code: unique_concepts.add(code)
        concept_map = {val: i+1 for i, val in enumerate(sorted(list(unique_concepts)))}

        df['uid_idx'] = df['user_id'].map(user_map)
        df['iid_idx'] = df['item_id'].map(item_map)

        self.config.num_students = len(user_map) + 1
        self.config.num_questions = len(item_map) + 1
        self.config.num_concepts = len(concept_map) + 1

        print(f"统计: 学生 {len(user_map)}, 题目 {len(item_map)}, 知识点 {len(concept_map)}")

        # 构建归一化 Q-Matrix
        q_matrix = np.zeros((self.config.num_questions, self.config.num_concepts))
        tmp_df = df[['iid_idx', 'knowledge_code']].drop_duplicates()
        for _, row in tmp_df.iterrows():
            cleaned = str(row['knowledge_code']).replace('[', '').replace(']', '').replace("'", "").replace('"', '')
            for code in cleaned.split(','):
                code = code.strip()
                if code in concept_map:
                    q_matrix[int(row['iid_idx']), concept_map[code]] = 1

        q_matrix_norm = q_matrix / np.maximum(q_matrix.sum(axis=1, keepdims=True), 1)
        item_diff = df.groupby('iid_idx')['score'].mean()
        diff_values = np.zeros(self.config.num_questions)
        for iid, diff in item_diff.items():
            diff_values[iid] = diff

        # 序列生成
        sequences, records = [], []
        u_list = df['uid_idx'].unique()[:1500]
        df_exp = df[df['uid_idx'].isin(u_list)].sort_index()

        for uid, group in tqdm(df_exp.groupby('uid_idx'), desc="处理序列"):
            if len(group) < 3: continue

            concept_names = []
            for codes in group['knowledge_code'].astype(str):
                cleaned = codes.replace('[', '').replace(']', '').replace("'", "").replace('"', '')
                concept_names.append(" & ".join([f"K{c.strip()}" for c in cleaned.split(',') if c.strip()]))

            sequences.append({
                'uid': int(uid),
                'iids': [int(i) for i in group['iid_idx'].values],
                'labels': [int(l) for l in group['score'].values],
                'concept_names': concept_names
            })
            for _, row in group.iterrows():
                records.append([int(row['uid_idx']), int(row['iid_idx']), int(row['score'])])

        # --- 新增: 以文件形式导出处理好的序列 ---
        export_path = os.path.join(self.config.data_dir, "processed_sequences.json")
        print(f"正在导出序列数据到: {export_path}")
        with open(export_path, 'w', encoding='utf-8') as f:
            json.dump(sequences, f, ensure_ascii=False, indent=4)
        print("序列导出完成。")

        return sequences, records, q_matrix_norm, diff_values, concept_map

# ==========================================
# 3. 诊断模型 (保持完整架构)
# ==========================================

class IRT(nn.Module):
    def __init__(self, n_user, n_item):
        super().__init__()
        self.theta = nn.Embedding(n_user, 1)
        self.beta = nn.Embedding(n_item, 1)
    def forward(self, uid, iid):
        return torch.sigmoid(self.theta(uid) - self.beta(iid)).squeeze()

class DINA(nn.Module):
    def __init__(self, n_user, n_item, n_skill, q_matrix):
        super().__init__()
        self.register_buffer('q_matrix', torch.tensor(q_matrix, dtype=torch.float32))
        self.mastery = nn.Embedding(n_user, n_skill)
        self.slip = nn.Embedding(n_item, 1)
        self.guess = nn.Embedding(n_item, 1)
    def forward(self, uid, iid):
        m = torch.sigmoid(self.mastery(uid))
        q = self.q_matrix[iid]
        eta = torch.prod(torch.pow(m, q), dim=-1, keepdim=True)
        return (torch.pow(1-torch.sigmoid(self.slip(iid)), eta) * torch.pow(torch.sigmoid(self.guess(iid)), 1-eta)).squeeze()

class NeuralCDM(nn.Module):
    def __init__(self, n_user, n_item, n_skill, q_matrix):
        super().__init__()
        self.register_buffer('q_matrix', torch.tensor(q_matrix, dtype=torch.float32))
        self.student_emb = nn.Embedding(n_user, n_skill)
        self.item_emb = nn.Embedding(n_item, n_skill)
        self.layer1 = nn.Linear(n_skill, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 1)
    def forward(self, uid, iid):
        s, i = torch.sigmoid(self.student_emb(uid)), torch.sigmoid(self.item_emb(iid))
        x = (s - i) * self.q_matrix[iid]
        x = torch.sigmoid(F.linear(x, torch.abs(self.layer1.weight), self.layer1.bias))
        x = torch.sigmoid(F.linear(x, torch.abs(self.layer2.weight), self.layer2.bias))
        return torch.sigmoid(F.linear(x, torch.abs(self.layer3.weight), self.layer3.bias)).squeeze()

class HIG_TCAN_CD(nn.Module):
    def __init__(self, config, q_matrix, diff_values):
        super().__init__()
        self.q_emb = nn.Embedding(config.num_questions, config.embed_dim)
        self.s_emb = nn.Embedding(config.num_students, config.embed_dim)
        self.k_emb = nn.Embedding(config.num_concepts, config.embed_dim)
        self.register_buffer('q_k_rel', torch.tensor(q_matrix, dtype=torch.float32))
        self.register_buffer('diffs', torch.tensor(diff_values, dtype=torch.float32))
        self.diff_proj = nn.Linear(1, config.embed_dim)
        self.input_proj = nn.Linear(config.embed_dim + 1, config.embed_dim)
        self.query = nn.Linear(config.embed_dim, config.embed_dim)
        self.key = nn.Linear(config.embed_dim, config.embed_dim)
        self.value = nn.Linear(config.embed_dim, config.embed_dim)
        self.pred_mlp = nn.Sequential(nn.Linear(config.embed_dim*3, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid())

    def forward(self, q_ids, labels, u_id):
        k_agg = torch.matmul(self.q_k_rel, self.k_emb.weight)
        q_feat = self.q_emb(q_ids) + F.embedding(q_ids, k_agg) + self.diff_proj(self.diffs[q_ids].unsqueeze(-1))
        x = self.input_proj(torch.cat([q_feat, labels.unsqueeze(-1)], dim=-1))
        B, L, D = x.size()
        Q, K, V = self.query(x), self.key(x), self.value(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(D)
        mask = torch.tril(torch.ones(L, L, device=x.device))
        h = torch.matmul(F.softmax(scores.masked_fill(mask == 0, -1e9), dim=-1), V)
        s_static = self.s_emb(u_id).unsqueeze(1).expand(-1, L, -1)
        feat = torch.cat([h[:, :-1, :], q_feat[:, 1:, :], s_static[:, 1:, :]], dim=-1)
        return self.pred_mlp(feat).squeeze(-1)

# ==========================================
# 4. LLM 评估与运行器
# ==========================================

class LLMEvaluator:
    def __init__(self, config):
        self.config = config

    def call_api(self, prompt):
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{self.config.llm_model}:generateContent?key={self.config.api_key}"
        payload = {
            "contents": [{"parts": [{"text": prompt}]}],
            "systemInstruction": {"parts": [{"text": "你是一个认知诊断专家。预测下一题对错。格式必须为 JSON。"}]},
            "generationConfig": {"responseMimeType": "application/json", "responseSchema": {"type": "OBJECT", "properties": {"reasoning": {"type": "string"}, "prediction": {"type": "integer"}}, "required": ["reasoning", "prediction"]}}
        }
        for i in range(5):
            try:
                req = urllib.request.Request(url, data=json.dumps(payload).encode(), headers={'Content-Type': 'application/json'})
                with urllib.request.urlopen(req, timeout=30) as res:
                    return json.loads(json.loads(res.read().decode())['candidates'][0]['content']['parts'][0]['text'])
            except: time.sleep(2**i)
        return {"reasoning": "API Error", "prediction": 0}

    def evaluate(self, test_samples):
        print("\n>>> LLM 推理评估...")
        preds, targets = [], []
        for i, sample in enumerate(tqdm(test_samples)):
            history = "\n".join([f"- {s}: {'对' if l==1 else '错'}" for s, l in zip(sample['concept_names'][-11:-1], sample['labels'][-11:-1])])
            res = self.call_api(f"历史:\n{history}\n预测知识点: {sample['concept_names'][-1]}")
            if i < self.config.showcase_count:
                print(f"\n[样例] 知识点: {sample['concept_names'][-1]}\n诊断: {res['reasoning']}\n结果: {res['prediction']} (真实: {sample['labels'][-1]})")
            preds.append(res['prediction']); targets.append(sample['labels'][-1])
        return roc_auc_score(targets, preds), accuracy_score(targets, preds)

class ExperimentRunner:
    def __init__(self, config):
        self.config = config
        self.results = {}

    def run(self, name, model, train_loader, test_loader, is_seq=False):
        model.to(self.config.device); opt = torch.optim.Adam(model.parameters(), lr=self.config.lr)
        print(f"\n>>> 训练 {name}...")
        for epoch in range(self.config.epochs):
            model.train()
            for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
                opt.zero_grad()
                if is_seq:
                    q, r, u = [b.to(self.config.device) for b in batch]
                    loss = F.binary_cross_entropy(model(q, r, u), r[:, 1:])
                else:
                    u, i, r = [b.to(self.config.device) for b in batch]
                    loss = F.binary_cross_entropy(model(u, i), r.float())
                loss.backward(); opt.step()

        model.eval(); all_p, all_r = [], []
        with torch.no_grad():
            for batch in test_loader:
                if is_seq:
                    q, r, u = [b.to(self.config.device) for b in batch]
                    all_p.extend(model(q, r, u).cpu().numpy().flatten()); all_r.extend(r[:, 1:].cpu().numpy().flatten())
                else:
                    u, i, r = [b.to(self.config.device) for b in batch]
                    all_p.extend(model(u, i).cpu().numpy()); all_r.extend(r.cpu().numpy())
        self.results[name] = {'AUC': roc_auc_score(all_r, all_p), 'ACC': accuracy_score(all_r, [1 if p>0.5 else 0 for p in all_p])}

def main():
    config = Config(); proc = Math2015Processor(config)
    try: seqs, recs, q_mat, diffs, _ = proc.process()
    except Exception as e: print(f"失败: {e}"); return

    runner = ExperimentRunner(config)
    train_r, test_r = train_test_split(recs, test_size=0.2, random_state=42)
    static_tr, static_te = DataLoader(train_r, batch_size=config.batch_size, shuffle=True), DataLoader(test_r, batch_size=config.batch_size)

    def collate_fn(batch):
        qs = [torch.tensor(x['iids'][:config.seq_len]) for x in batch]
        rs = [torch.tensor(x['labels'][:config.seq_len]) for x in batch]
        return torch.nn.utils.rnn.pad_sequence(qs, batch_first=True), torch.nn.utils.rnn.pad_sequence(rs, batch_first=True).float(), torch.tensor([x['uid'] for x in batch])

    train_s, test_s = train_test_split(seqs, test_size=0.2, random_state=42)
    seq_tr, seq_te = DataLoader(train_s, batch_size=config.batch_size, collate_fn=collate_fn, shuffle=True), DataLoader(test_s, batch_size=config.batch_size, collate_fn=collate_fn)

    runner.run("IRT", IRT(config.num_students, config.num_questions), static_tr, static_te)
    runner.run("DINA", DINA(config.num_students, config.num_questions, config.num_concepts, q_mat), static_tr, static_te)
    runner.run("NeuralCDM", NeuralCDM(config.num_students, config.num_questions, config.num_concepts, q_mat), static_tr, static_te)
    runner.run("HIG-TCAN_CD", HIG_TCAN_CD(config, q_mat, diffs), seq_tr, seq_te, is_seq=True)

    l_auc, l_acc = LLMEvaluator(config).evaluate(random.sample(test_s, min(len(test_s), config.llm_sample_size)))
    runner.results['LLM (Gemini)'] = {'AUC': l_auc, 'ACC': l_acc}

    print("\n" + "="*45); print(f"{'Model':<18} | {'AUC':<10} | {'ACC':<10}"); print("-" * 45)
    for name, m in runner.results.items(): print(f"{name:<18} | {m['AUC']:.4f}     | {m['ACC']:.4f}")
    print("="*45)

if __name__ == "__main__": main()

正在下载 math2015...


downloader, INFO http://staff.ustc.edu.cn/~qiliuql/data/math2015.rar is saved as data/math2015/math2015.rar


downloader, INFO data/math2015/math2015.rar is unrar to data/math2015/math2015



找到数据文件: ./data/math2015/math2015/math2015/Math2/data.txt
检测到稠密矩阵格式，正在转换为三元组...
找到 Q 矩阵文件: ./data/math2015/math2015/math2015/Math2/qnames.txt
统计: 学生 3911, 题目 17, 知识点 0


处理序列: 100%|██████████| 1500/1500 [00:02<00:00, 596.01it/s]


...


[样例] 知识点: 
诊断: API Error
结果: 0 (真实: 1)


 10%|█         | 2/20 [01:03<09:33, 31.87s/it]


[样例] 知识点: 
诊断: API Error
结果: 0 (真实: 0)


100%|██████████| 20/20 [10:34<00:00, 31.74s/it]


Model              | AUC        | ACC       
---------------------------------------------
IRT                | 0.6427     | 0.5949
DINA               | 0.7113     | 0.6659
NeuralCDM          | 0.5000     | 0.5329
HIG-TCAN_CD        | 0.7361     | 0.6667
LLM (Gemini)       | 0.5000     | 0.7000
